# Turkish Earthquake Discourse Frame Detection — Colab Quickstart

End-to-end run: install deps → mount Drive → TF-IDF baseline → BERTurk fine-tune → score full corpus.

**Before running:** set Runtime → Change runtime type → **GPU (T4 is fine, A100 if available)**.

**Drive layout expected:**
```
MyDrive/deprem/
  annotation/final_300.csv
  corpus/corpus_final.csv
MyDrive/nlp_pipeline/         <- upload the .py files here (config.py, pipeline_*.py, predict_corpus.py)
```

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.18" "accelerate>=0.27" scikit-learn pandas numpy

## 2. Mount Google Drive and add the pipeline folder to sys.path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/nlp_pipeline')

# Sanity check: confirm files are visible
import os
for p in [
    '/content/drive/MyDrive/nlp_pipeline/config.py',
    '/content/drive/MyDrive/deprem/annotation/final_300.csv',
    '/content/drive/MyDrive/deprem/corpus/corpus_final.csv',
]:
    print(('OK   ' if os.path.exists(p) else 'MISS '), p)

## 3. Confirm GPU is available

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 4. TF-IDF + LinearSVR baseline (~30 seconds)

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_tfidf_svr.py

## 5. BERTurk fine-tune (~10–20 min on T4)

Saves the model to `MyDrive/deprem/outputs/berturk_model/`.

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python pipeline_berturk.py

## 6. Score the full 2139-doc corpus

Writes `MyDrive/deprem/outputs/analytic_with_scores.csv`.

In [ ]:
%cd /content/drive/MyDrive/nlp_pipeline
!python predict_corpus.py

## 7. Peek at the scored corpus

In [ ]:
import pandas as pd
scored = pd.read_csv('/content/drive/MyDrive/deprem/outputs/analytic_with_scores.csv', encoding='utf-8-sig')
print(scored.shape)
scored[['doc_id','province','score_technical','score_political','score_development','score_sustainability']].head(10)